# Functions Notebook
This notebook holds different functions of the solution at it's current level in development. <br> 
This notebook is provided to quickly set-up the developer environment, as well as, demonstrate the capabilities of the current system. 

There are optional steps within this functions notebook, as it may not be required for your installation to further train the model, or create a new model. <br> 
If you are not in a position where you will be assisting with the training of models, ignore functions (or blocks) after "1". 

**Required setup code-blocks:**
- 0 - Installing required dependencies
- 1 - Download models from HuggingFace

Optional functional code-blocks:
- 2 - Download image dataset from Roboflow
- 3 - Train YOLOv8x model with downloaded dataset
- 4 - Upload model weights to HuggingFace



## 0 - Installing required dependencies
The following codeblock will install the required packages for our codebase. <br>
It is recommended that the installation of packages is done within a localised venv.<br>
This is required for all users of the solution

In [1]:
%pip install -r requirements.txt

   ---------------------------------------- 0.0/974.8 kB ? eta -:--:--
   ---------------------------------------- 974.8/974.8 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.3.107
    Uninstalling ultralytics-8.3.107:
      Successfully uninstalled ultralytics-8.3.107
Note: you may need to restart the kernel to use updated packages.


## 1 - Download models from HuggingFace
The following codeblock works to fetch the inference models hosted on HuggingFace. <br>
This codeblock needs to be run prior to attempting to use any of the scripts in this repo.



In [2]:
import os
from huggingface_hub import hf_hub_download, list_repo_files

# Setup
repo_id = "ast-n/cv4gt"
local_model_dir = "models"
os.makedirs(local_model_dir, exist_ok=True)

# Filter search for all models only
model_files = [f for f in list_repo_files(repo_id) if f.endswith(".pt")]

# Download to models/
for model_name in model_files:
    try:
        model_path = hf_hub_download(repo_id=repo_id, filename=model_name, local_dir=local_model_dir)
        print(f"Downloaded: {model_name} -> {model_path}")
    except Exception as e:
        print(f"Failed to download {model_name}: {e}")


Downloaded: YOLOv8-cv4gt-bad-dataset_epochs500.pt -> models\YOLOv8-cv4gt-bad-dataset_epochs500.pt


YOLOv8-cv4gt-data-11-04_100e.pt:   0%|          | 0.00/137M [00:00<?, ?B/s]

Downloaded: YOLOv8-cv4gt-data-11-04_100e.pt -> models\YOLOv8-cv4gt-data-11-04_100e.pt


YOLOv8-cv4gt-data-15-04_100e.pt:   0%|          | 0.00/137M [00:00<?, ?B/s]

Downloaded: YOLOv8-cv4gt-data-15-04_100e.pt -> models\YOLOv8-cv4gt-data-15-04_100e.pt


## 2 - Download image dataset from Roboflow
The following codeblock works to fetch the newest version of the image dataset from our Roboflow project. <br>
This codeblock needs to be ran and the dataset needs to successfully be placed in `'data/training'` for training to be conducted. <br>
If you are downloading different versions of the dataset on the same day, you'll need to remove this before downloading.


In [3]:
import getpass
import os
from datetime import datetime
from roboflow import Roboflow

# Generate date for consistent naming of datasets
today_str = datetime.today().strftime("%d-%m")
dataset_name = f"cv4gt-data-{today_str}"
save_path = os.path.join("data", "training", dataset_name)

# Get Roboflow auth
api_key = getpass.getpass("Enter your Roboflow API key: \n")
rf = Roboflow(api_key)

# Access project, grab latest 
project = rf.workspace("cos40005capstone").project("cv4gt-data")
latest_version = project.versions()[0].version
version = project.version(latest_version)

# Download dataset
dataset = version.download("yolov8", location=save_path)

Enter your Roboflow API key: 
 ········


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data\training\cv4gt-data-16-04 in yolov8:: 100%|█| 2882/2882 [00:02<00:00, 1373.77it/


## 3 - Training a custom model
The following codeblock runs off of the logic found in the `train_test.py` file. It imports functions `train_model` and `save_model`. <br>
Initiation of this codeblock will begin running the training of this dataset. 



In [ ]:
import sys
sys.path.append("src")
from train_test import train_model, save_model

# Run the train_model function at designated epochs amount
train_model(epochs=100)

### Save the trained model
Saves the `best.pt` weight from the last created `train*` folder in `runs/detect/train*` and copies this to `models/` with a formatted name.

In [ ]:
# Wait until training finishes
save_model(epochs=100)

## 4 - Upload trained model weights to HuggingFace
This codeblock is to be used to upload newly trained models to the HuggingFace repo. <br>
A HuggingFace access token is required to be entered in-order to validate user access to the repo.  

In [ ]:
import os
import getpass
from huggingface_hub import HfApi

# Setup 
hf_token = getpass.getpass("Enter your HuggingFace access token: \n")
repo_id = "ast-n/cv4gt"
local_model_dir = "models"
api = HfApi()

# Upload all new model files
for filename in os.listdir(local_model_dir):
    if filename.endswith(".pt"):
        local_path = os.path.join(local_model_dir, filename)
        try:
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=filename,
                repo_id=repo_id,
                repo_type="model",
                token=hf_token
            )
            print(f"Uploaded: {filename}")
        except Exception as e:
            print(f"Error uploading {filename}: {e}")


